# W27 · RL 策略 × ROS2：策略节点封装

> 这是整个 12 个月项目的「会师点」：阶段一/二训练的 SB3 / MotrixLab 策略，
> 从 notebook 里的 `model.predict(obs)` 变成机器人系统里一个**可插拔的 ROS2 节点**。
> 本讲给出完整、可移植的工程模板。

## 学习目标

1. 说清「训练侧契约」（观测定义、归一化、动作含义、控制频率）为什么必须在部署侧逐字节复现；
2. 掌握分层设计：**纯 Python 的 PolicyCore（可单元测试）** + **薄 ROS 适配层**；
3. 会用参数化方式加载 SB3 模型、订阅观测、定频发布动作；
4. 会给策略节点加安全机制：watchdog 超时停车、动作限幅、异常兜底；
5. 理解策略节点与 Nav2/ros2_control 的接口关系。

## ⚠️ 运行前提

PolicyCore 演示为纯 Python（numpy），本机真实执行；完整 rclpy 节点模板需 ROS2 Jazzy
+ SB3 环境（在 Ubuntu 24.04 上用 `pip install stable-baselines3` 装进用户环境即可，
注意与系统 Python 的兼容）。

## 1. 训练→部署的最大陷阱：契约漂移

在 Gymnasium 里，`env.step(action)` 的 `obs` 由环境亲手喂给你，格式永远正确。
部署到 ROS2 后，**你要自己从消息里拼 obs**——任何一个环节不一致，策略就会失效，
而且往往**不报错、只是表现变差**（最难排查的一类 bug）：

| 契约项 | 训练侧 | 部署侧常见错误 |
|--------|--------|----------------|
| 观测顺序 | `[h_err, v]` | 从两个消息拼接时顺序写反 |
| 单位/坐标系 | m, m/s, 机体坐标系 | 消息是世界系、IMU 是四元数没转欧拉角 |
| 归一化 | VecNormalize 的 running mean/std | 忘加载归一化统计量，或加载错版本 |
| 控制频率 | `dt=0.02`（50 Hz） | 定时器 10 Hz，动作被「拉伸」5 倍 |
| 动作含义 | [-1,1] → [0, 2g] 推力 | 直接当速度/位置指令发给控制器 |

**工程对策**：把契约**代码化**——观测打包、归一化、动作后处理写成纯函数，
训练侧和部署侧 import 同一份代码（或至少同一组单元测试）。

## 2. 分层设计：PolicyCore 与 ROS 适配层

```
   ┌────────── ROS 适配层（rclpy，不可单测，靠集成测试）──────────┐
   │  订阅 /odom,/imu ──▶ msg 转 dict ──▶ PolicyCore ──▶ 发布 /cmd_vel │
   └──────────────────────────────────────────────────────────────┘
                     ▲ 纯 Python，pytest 可测 ▲
   ┌──────────────── PolicyCore ────────────────────────────────┐
   │  build_obs(sensor_dict) -> np.ndarray                       │
   │  predict(obs) -> action（内部调用 SB3 模型 + 归一化 + 限幅） │
   └─────────────────────────────────────────────────────────────┘
```

下面实现一个**最小但完整**的 PolicyCore（用线性策略代替 SB3，演示全部契约逻辑，
本机执行）。真实部署时只需把 `_linear_forward` 换成 `self._model.predict(obs, deterministic=True)`：

In [1]:
"""PolicyCore：与 ROS 解耦的策略核心（纯 Python，本机执行）。

契约（以阶段一 DroneHoverEnv 为例）：
  obs = [height_err, vertical_vel]，单位 m / m·s⁻¹，机体坐标系；
  action ∈ [-1, 1]，映射到 [0, 2g] 推力；控制周期 dt = 0.02 s。
"""
from dataclasses import dataclass, field

import numpy as np

OBS_KEYS = ("height_err", "vertical_vel")   # 观测顺序 = 契约，训练/部署共用此常量
OBS_CLIP = np.array([5.0, 10.0], dtype=np.float32)   # 与 observation_space 一致
DT = 0.02


@dataclass
class PolicyCore:
    """可单元测试的策略核心。SB3 部署时把 linear 换成 model.predict。"""

    # 线性演示策略：err = target - h > 0（高度偏低）→ 加推力；v > 0（上升中）→ 收油门。
    # 权重符号与契约定义强相关——写错一个符号闭环就发散（练习 2 会让你亲手验证）。
    weights: np.ndarray = field(default_factory=lambda: np.array([0.6, -0.35]))
    bias: float = 0.0
    # 归一化统计量（训练侧 VecNormalize 导出；此处演示用恒等）
    obs_mean: np.ndarray = field(default_factory=lambda: np.zeros(2))
    obs_std: np.ndarray = field(default_factory=lambda: np.ones(2))

    def build_obs(self, sensors: dict) -> np.ndarray:
        """从传感器字典按契约顺序打包观测。缺字段必须显式报错，不许静默补零。"""
        missing = [k for k in OBS_KEYS if k not in sensors]
        if missing:
            raise KeyError(f"观测缺字段: {missing}")
        obs = np.array([sensors[k] for k in OBS_KEYS], dtype=np.float32)
        return np.clip(obs, -OBS_CLIP, OBS_CLIP)

    def normalize(self, obs: np.ndarray) -> np.ndarray:
        return (obs - self.obs_mean) / self.obs_std

    def act(self, sensors: dict) -> float:
        """端到端：sensors -> 标量动作（已限幅到 [-1, 1]）。"""
        obs = self.normalize(self.build_obs(sensors))
        a = float(obs @ self.weights + self.bias)   # 真实部署: self._model.predict(obs, deterministic=True)[0]
        return float(np.clip(a, -1.0, 1.0))


# --- 单元测试风格的验证（部署前这些断言必须在 pytest 里常驻）---
core = PolicyCore()
a = core.act({"height_err": 1.0, "vertical_vel": 0.0})
assert a > 0, "err>0（高度偏低）时必须加推力（action>0）——符号契约测试"
print(f"符号契约 OK: err=+1.0 -> action={a:+.2f}（加推力，符合 err = target - h 的定义）")

# 闭环冒烟：用 DroneHoverEnv 的动力学近似验证策略能收敛
h, v = 0.2, 0.0
for step in range(200):
    a = core.act({"height_err": 1.0 - h, "vertical_vel": v})
    acc = (a + 1.0) * 9.81 - 9.81          # action [-1,1] -> 推力加速度 [0, 2g]，减重力
    v += acc * DT
    h += v * DT
print(f"闭环 200 步后: h={h:.3f} m（目标 1.0）, v={v:+.3f} m/s -> 收敛到目标高度附近")

符号契约 OK: err=+1.0 -> action=+0.60（加推力，符合 err = target - h 的定义）
闭环 200 步后: h=0.999 m（目标 1.0）, v=+0.001 m/s -> 收敛到目标高度附近


**观察**：线性演示策略在闭环中收敛到目标高度。真正重要的不是这个权重，
而是第一个 `assert`——它把「err 定义 → 推力方向」的符号契约固化成了测试：
部署侧任何人改 `build_obs`、改误差定义、改动作映射，测试立刻红灯。
SB3 部署版本只需把线性前向换成：

```python
from stable_baselines3 import PPO
self._model = PPO.load(policy_path)          # zip 内含网络权重与归一化（若保存了 VecNormalize）
action, _ = self._model.predict(obs, deterministic=True)   # 部署永远用确定性策略
```

## 3. 完整 rclpy 策略节点模板

下面这份模板可直接复制进你的 ROS2 包。设计要点：

- **参数化一切**：模型路径、话题名、控制频率全部走 parameter（W22）；
- **定频控制**：用 timer 而非「收到观测就推理」——观测异步到达，控制必须等周期（W22 的 Executor 课）；
- **watchdog**：观测超时立刻输出安全动作并告警（W25 的 `cmd_vel_timeout` 是最后一道防线，策略节点自身是第一道）；
- **异常兜底**：推理抛错时发布安全动作，而不是让节点崩掉。

In [ ]:
# ⚠️ 运行前提：需要 ROS2 Jazzy 环境（Ubuntu 24.04 + apt 安装，见 docs/phase3_ros2.md）。
# 本机没有 ROS2，此 cell 仅作模板，保持未执行状态；请复制到 ROS2 工作区中运行。
"""rl_policy_node.py —— 把 SB3 策略部署为 ROS2 节点（完整可移植模板）。

依赖：rclpy、stable_baselines3、numpy。
运行：
    ros2 run my_rl_pkg rl_policy_node --ros-args         -p policy_path:=/path/to/ppo_hover.zip -p control_rate:=50.0
接口：
    订阅: /hover/height (std_msgs/Float64), /hover/vertical_vel (std_msgs/Float64)
    发布: /hover/thrust  (std_msgs/Float64, 范围 [-1,1]，下游控制器负责映射到电机)
"""
import time

import numpy as np
import rclpy
from rclpy.node import Node
from std_msgs.msg import Float64

OBS_KEYS = ("height_err", "vertical_vel")   # 与训练侧契约一致
OBS_CLIP = np.array([5.0, 10.0], dtype=np.float32)


class RLPolicyNode(Node):
    def __init__(self):
        super().__init__("rl_policy_node")
        # ---- 参数：部署契约的全部可调项 ----
        self.declare_parameter("policy_path", "")
        self.declare_parameter("control_rate", 50.0)      # 必须等于训练 dt 的倒数
        self.declare_parameter("obs_timeout", 0.2)        # watchdog：观测陈旧阈值 (s)
        self.declare_parameter("height_topic", "/hover/height")
        self.declare_parameter("vel_topic", "/hover/vertical_vel")
        self.declare_parameter("action_topic", "/hover/thrust")

        policy_path = self.get_parameter("policy_path").value
        if not policy_path:
            raise ValueError("必须通过 -p policy_path:=... 指定模型文件")

        # ---- 加载 SB3 模型（延迟导入，让无 GPU/无 SB3 的机器也能启动报错友好）----
        from stable_baselines3 import PPO
        self._model = PPO.load(policy_path)
        self.get_logger().info(f"策略已加载: {policy_path}")

        # ---- 观测缓存：消息异步到达，控制周期同步消费 ----
        self._obs_buf: dict[str, tuple[float, float]] = {}  # key -> (值, 接收时刻)
        self._action_pub = self.create_publisher(
            Float64, self.get_parameter("action_topic").value, 10)
        self.create_subscription(
            Float64, self.get_parameter("height_topic").value,
            lambda m: self._on_obs("height_err", m), 10)
        self.create_subscription(
            Float64, self.get_parameter("vel_topic").value,
            lambda m: self._on_obs("vertical_vel", m), 10)

        rate = self.get_parameter("control_rate").value
        self.create_timer(1.0 / rate, self._on_control_tick)
        self.get_logger().info(f"控制循环启动: {rate:.0f} Hz")

    def _on_obs(self, key: str, msg: Float64) -> None:
        self._obs_buf[key] = (float(msg.data), time.monotonic())

    def _obs_is_fresh(self) -> bool:
        timeout = self.get_parameter("obs_timeout").value
        now = time.monotonic()
        return (
            all(k in self._obs_buf for k in OBS_KEYS)
            and all(now - t < timeout for _, t in self._obs_buf.values())
        )

    def _publish_safe_action(self, reason: str) -> None:
        # 安全动作：0 推力（等价于「不注入新能量」，具体语义按机器人定义）
        self._action_pub.publish(Float64(data=0.0))
        self.get_logger().warn(f"安全动作已发布: {reason}", throttle_duration_sec=1.0)

    def _on_control_tick(self) -> None:
        # ---- watchdog：观测缺失/陈旧 -> 安全动作 ----
        if not self._obs_is_fresh():
            self._publish_safe_action("观测超时或缺失")
            return
        # ---- 打包观测（契约：顺序、clip、float32）----
        obs = np.array([self._obs_buf[k][0] for k in OBS_KEYS], dtype=np.float32)
        obs = np.clip(obs, -OBS_CLIP, OBS_CLIP)
        # ---- 推理（确定性；耗时应远小于控制周期，见 W22 Executor 课）----
        try:
            action, _ = self._model.predict(obs, deterministic=True)
        except Exception as exc:  # 推理异常兜底
            self._publish_safe_action(f"推理异常: {exc}")
            return
        self._action_pub.publish(Float64(data=float(np.clip(action[0], -1.0, 1.0))))


def main(args=None):
    rclpy.init(args=args)
    node = RLPolicyNode()
    try:
        rclpy.spin(node)
    except KeyboardInterrupt:
        pass
    finally:
        node.destroy_node()
        rclpy.shutdown()

**预期行为**：

- 正常时 `/hover/thrust` 以 50 Hz 输出动作；`ros2 topic hz` 验证频率达标；
- 停掉任一观测发布者后 0.2 s 内，动作归零并打印 `观测超时或缺失`；
- 推理耗时可用 `ros2 topic delay /hover/thrust` 或日志计时检查，
  **P50 应远小于 20 ms**——CPU 上的 MlpPolicy 通常在 1 ms 量级，绰绰有余。

## 4. 与 Nav2 / ros2_control 的接口关系

三种集成深度（W28 项目会用到第 1、2 种）：

1. **策略替代 controller_server**：Nav2 全局规划出路径，你的 RL 策略订阅
   `/plan` + 局部观测，直接发布 `/cmd_vel` → ros2_control 执行；
2. **策略作为局部安全层**：Nav2 的 `/cmd_vel` 与策略输出经 `twist_mux` 仲裁，
   急停/避障时策略通道优先级更高；
3. **策略即底层控制器**（四旋翼常见）：策略直接输出电机级指令/推力，
   跳过 Nav2——此时 watchdog 与安全动作是生死线。

## ✏️ 练习

### 练习 1（★，约 15 分钟，纯 Python）：契约测试

为本讲 `PolicyCore` 写 4 个断言并运行：(a) 缺字段抛 `KeyError`；(b) 超范围观测被 clip；
(c) 输出恒在 [-1,1]；(d) 两次调用同样输入输出相同（确定性）。交付：测试代码 + 全部通过的输出。

### 练习 2（★★，约 30 分钟，纯 Python）：契约漂移的代价

用本讲的闭环冒烟代码做对照实验：故意 (a) 交换 obs 两个分量；(b) 把 action 缩放 0.5 倍；
(c) 把 DT 从 0.02 改成 0.1（模拟频率错配）。记录每种情况下 200 步后的高度。
交付：对照表格 + 一句话解释为什么这类 bug 在真机上「不报错但表现差」。

### 练习 3（★★，约 40 分钟）：部署 DroneHoverEnv 的 PPO 策略

在有 SB3 的机器上：用阶段一的 PPO 在 `DroneHoverEnv` 训练一个能悬停的策略并保存 zip；
写一个 `hover_sim_node`（以 50 Hz 发布模拟的 height/vel，内置简化动力学）+
本讲的 `rl_policy_node`，在纯 ROS2 环境里闭环跑 10 s。
交付：训练命令、两节点代码、`ros2 topic echo /hover/thrust` 片段 + 高度收敛日志。

### 练习 4（★★★，约 50 分钟）：watchdog 分级

把节点的安全策略升级为三级：观测延迟 < 0.2 s 正常推理；0.2–1.0 s 用「最后有效动作 × 衰减系数」；
> 1.0 s 输出安全动作并请求上层恢复（发布 `/policy_node/emergency` Bool）。
交付：代码 + 用 `ros2 topic pub --once` 手动断流触发的完整日志 + 状态转移图。

## 参考答案

<details>
<summary>参考答案</summary>

**练习 1**：

```python
core = PolicyCore()
# (a)
try:
    core.act({"height_err": 0.5})
    raise AssertionError("应抛 KeyError")
except KeyError:
    pass
# (b)
assert np.isclose(core.build_obs({"height_err": 99.0, "vertical_vel": 0})[0], 5.0)
# (c)
for s in [{"height_err": 5, "vertical_vel": 10}, {"height_err": -5, "vertical_vel": -10}]:
    assert -1.0 <= core.act(s) <= 1.0
# (d)
s = {"height_err": 0.3, "vertical_vel": 0.1}
assert core.act(s) == core.act(s)
```

**练习 2**：(a) 交换分量 → 策略把速度当位置误差，通常振荡或缓慢漂移；
(b) action ×0.5 → 增益不足，200 步后高度远未到位（欠驱动）；
(c) DT 错配 → 相当于控制周期变了 5 倍，积分发散或极限环。
共同点：每一步计算都「合法」，无任何异常——只有闭环行为不对。
这正是契约必须写成单元测试的原因。

**练习 3**（要点）：训练用 `PPO("MlpPolicy", env, verbose=1).learn(50_000)` 即可收敛；
`hover_sim_node` 内嵌 `v += ((a+1)*g - g)*dt; h += v*dt`，以 timer 50 Hz 发布两个 Float64；
注意 sim 节点与 policy 节点都要在同频率，否则会出现「动作被重复应用 N 次」的隐性频率错配。

**练习 4**（状态机要点）：

```python
age = now - min(t for _, t in self._obs_buf.values())
if age < 0.2:
    self._state = "ACTIVE"
elif age < 1.0:
    self._state = "DEGRADED"   # action = last_action * exp(-(age-0.2)/0.3)
else:
    self._state = "EMERGENCY"  # 安全动作 + 发布 /policy_node/emergency
```

关键细节：从 EMERGENCY 恢复需要观测重新连续 fresh 若干拍（迟滞），
防止在边界抖动导致动作突变。
</details>

## 延伸阅读

- [SB3 文档：Saving/Loading 与 predict](https://stable-baselines3.readthedocs.io/en/master/guide/save_format.html)
- [ROS2 设计文档：Actions](https://design.ros2.org/articles/actions.html)（把「任务」暴露成接口的思路，策略节点进阶版可以做成 Action Server）
- [rclpy 示例仓库](https://github.com/ros2/examples)（参数、QoS、executor 的最小例子）
- 下一讲预告：W28 阶段项目——把前七讲的零件装成一台自主导航机器人。